# KV 캐시: 효율적인 자기회귀 추론 - KV 캐시로 추론 가속화

- Tutorial ID: `adv-4-1`
- Tutorial: KV 캐시: 효율적인 자기회귀 추론
- Section ID: `adv-4-1-1`
- Section: KV 캐시로 추론 가속화


In [ ]:
# ============================================================
# 코드 읽는 법 — KV 캐시로 추론 가속화
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   3) autoregressive decoding에서 이전 K/V를 재사용해 계산을 줄이는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("KV 캐시: 자기회귀 추론 가속화")
print("=" * 60)

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

d_model = 64
d_head = 16
n_heads = 4
np.random.seed(42)

W_Q = np.random.randn(d_model, n_heads * d_head) * 0.1
W_K = np.random.randn(d_model, n_heads * d_head) * 0.1
W_V = np.random.randn(d_model, n_heads * d_head) * 0.1

In [ ]:
# 1. 캐시 없이 vs 있을 때 FLOPS 비교

In [ ]:
print("\n1. 캐시 효과 비교")
print("-" * 45)

seq_len = 10
embeddings = np.random.randn(seq_len, d_model)

# 캐시 없이
total_no_cache = 0
for t in range(1, seq_len + 1):
    flops = t * d_model * (n_heads * d_head) * 3 + t * t * (n_heads * d_head)
    total_no_cache += flops

# 캐시 있으면
total_cache = 0
for t in range(1, seq_len + 1):
    flops = d_model * (n_heads * d_head) * 3 + t * (n_heads * d_head)
    total_cache += flops

print(f"시퀀스 길이: {seq_len}")
print(f"총 FLOPS (캐시 없음): {total_no_cache:,}")
print(f"총 FLOPS (캐시 사용): {total_cache:,}")
print(f"속도 향상: {total_no_cache / total_cache:.2f}x")

In [ ]:
# 2. KV 캐시 구현

In [ ]:
print("\n2. KV 캐시 구현")
print("-" * 45)

# KV 캐시를 딕셔너리로 관리 (class 없이)
def create_kv_cache():
    return {'K': None, 'V': None}

def update_cache(cache, new_K, new_V):
    if cache['K'] is None:
        cache['K'] = new_K
        cache['V'] = new_V
    else:
        cache['K'] = np.vstack([cache['K'], new_K])
        cache['V'] = np.vstack([cache['V'], new_V])
    return cache

cache = create_kv_cache()
for t in range(5):
    x_new = embeddings[t:t+1]
    K_new = x_new @ W_K
    V_new = x_new @ W_V
    cache = update_cache(cache, K_new, V_new)
    print(f"  step {t}: cache K shape = {cache['K'].shape}")

In [ ]:
# 3. 메모리 분석 (LLaMA-7B)

In [ ]:
print("\n3. KV 캐시 메모리 (LLaMA-7B)")
print("-" * 45)

n_layers = 32
n_heads = 32
d_head = 128
max_seq = 4096

for method, n_kv in [("MHA", 32), ("GQA-8", 8), ("MQA", 1)]:
    size = 2 * n_layers * max_seq * n_kv * d_head * 2  # FP16
    print(f"  {method:6s} ({n_kv:2d} KV heads): {size/1e9:.2f} GB")

print("\n양자화 효과:")
for prec, bpp in [("FP16", 2), ("INT8", 1), ("INT4", 0.5)]:
    size = 2 * n_layers * max_seq * n_heads * d_head * bpp
    print(f"  {prec}: {size/1e9:.2f} GB")